# Module 05: Engineer Data for Predictive Modeling with BigQuery ML

**CareerAlign GCP Professional ML Engineer**

This notebook covers:
1. Apache Beam / Dataflow pipeline (local runner)
2. BigQuery: derived features with SQL
3. Handling missing values and class imbalance
4. Partitioned table creation and query optimization
5. BQML model: engineered features vs raw features

> **Note**: Cells marked with `# REQUIRES GCP` need a GCP project with BigQuery enabled. Local-only cells run anywhere.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-mle/notebooks/05-data-engineering-bqml.ipynb)

---
## Setup & Installations

In [ ]:
# Install required packages
!pip install -q apache-beam[gcp] pandas numpy scikit-learn google-cloud-bigquery db-dtypes

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

---
## 1. Apache Beam Pipeline (Local DirectRunner)

This section demonstrates core Apache Beam concepts:
- **Pipeline**: The container for the entire workflow
- **PCollection**: Immutable distributed dataset
- **ParDo / DoFn**: Per-element parallel transforms
- **GroupByKey**: Grouping elements by key
- **Map / FlatMap**: Simple element-wise transforms

We'll run locally with `DirectRunner`. On GCP, you'd switch to `DataflowRunner`.

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

# --- Create sample CSV data locally ---
sample_csv = """user_001,purchase,45.99,2024-01-15 10:30:00
user_002,browse,0.0,2024-01-15 11:00:00
user_001,purchase,23.50,2024-01-15 14:20:00
user_003,purchase,,2024-01-16 09:00:00
user_002,purchase,89.99,2024-01-16 10:15:00
,browse,12.00,2024-01-16 11:30:00
user_001,browse,0.0,2024-01-17 08:45:00
user_003,purchase,150.00,2024-01-17 13:00:00
user_004,purchase,-5.00,2024-01-17 15:00:00
user_002,purchase,34.50,2024-01-18 09:30:00"""

with open('/tmp/beam_sample.csv', 'w') as f:
    f.write(sample_csv)

print("Sample CSV created with intentional data quality issues:")
print("  - Missing amount (row 4)")
print("  - Missing user_id (row 6)")
print("  - Negative amount (row 9)")

In [ ]:
# --- Define DoFn transforms ---

class ParseCSV(beam.DoFn):
    """Parse CSV lines into dictionaries. Handle missing fields."""
    def process(self, line):
        fields = line.strip().split(',')
        if len(fields) >= 4:
            yield {
                'user_id': fields[0] if fields[0] else None,
                'event_type': fields[1],
                'amount': float(fields[2]) if fields[2] else None,
                'timestamp': fields[3]
            }


class FilterInvalidRecords(beam.DoFn):
    """Remove records with missing user_id or negative amounts."""
    def process(self, record):
        if record['user_id'] and (record['amount'] is None or record['amount'] >= 0):
            yield record


class ImputeMissingAmount(beam.DoFn):
    """Impute missing amounts with 0.0 and add a flag."""
    def process(self, record):
        record['amount_is_missing'] = 1 if record['amount'] is None else 0
        record['amount'] = record['amount'] if record['amount'] is not None else 0.0
        yield record


class AddDerivedFeatures(beam.DoFn):
    """Add time-based derived features."""
    def process(self, record):
        ts = datetime.strptime(record['timestamp'], '%Y-%m-%d %H:%M:%S')
        record['day_of_week'] = ts.strftime('%A')
        record['hour_of_day'] = ts.hour
        record['is_weekend'] = 1 if ts.weekday() >= 5 else 0
        yield record


print("DoFn transforms defined:")
print("  1. ParseCSV - parse raw CSV lines")
print("  2. FilterInvalidRecords - remove bad records")
print("  3. ImputeMissingAmount - handle nulls")
print("  4. AddDerivedFeatures - time-based features")

In [ ]:
# --- Run the pipeline with DirectRunner ---

results = []

class CollectResults(beam.DoFn):
    """Collect results for display (local only - not for production)."""
    def process(self, element):
        results.append(element)
        yield element

# Build and run the pipeline
with beam.Pipeline(options=PipelineOptions()) as p:
    cleaned = (
        p
        | 'ReadCSV' >> beam.io.ReadFromText('/tmp/beam_sample.csv')
        | 'Parse' >> beam.ParDo(ParseCSV())
        | 'FilterInvalid' >> beam.ParDo(FilterInvalidRecords())
        | 'ImputeAmounts' >> beam.ParDo(ImputeMissingAmount())
        | 'AddFeatures' >> beam.ParDo(AddDerivedFeatures())
        | 'Collect' >> beam.ParDo(CollectResults())
    )

print(f"\nPipeline complete! Processed {len(results)} valid records from 10 raw rows.")
print(f"Filtered out {10 - len(results)} invalid records.\n")

# Display results
df_results = pd.DataFrame(results)
df_results

In [ ]:
# --- GroupByKey: Aggregate by user ---

user_aggregates = []

class CollectAgg(beam.DoFn):
    def process(self, element):
        user_aggregates.append(element)
        yield element

with beam.Pipeline(options=PipelineOptions()) as p:
    (
        p
        | 'ReadCSV' >> beam.io.ReadFromText('/tmp/beam_sample.csv')
        | 'Parse' >> beam.ParDo(ParseCSV())
        | 'Filter' >> beam.ParDo(FilterInvalidRecords())
        | 'Impute' >> beam.ParDo(ImputeMissingAmount())
        # Create key-value pairs for GroupByKey
        | 'ToKV' >> beam.Map(lambda r: (r['user_id'], r['amount']))
        | 'GroupByUser' >> beam.GroupByKey()
        # Compute aggregates per user
        | 'Aggregate' >> beam.Map(
            lambda kv: {
                'user_id': kv[0],
                'total_spend': sum(kv[1]),
                'num_events': len(kv[1]),
                'avg_spend': sum(kv[1]) / len(kv[1]) if kv[1] else 0
            }
        )
        | 'Collect' >> beam.ParDo(CollectAgg())
    )

print("GroupByKey results - Per-user aggregates:")
pd.DataFrame(user_aggregates)

---
## 2. BigQuery: Create Derived Features with SQL

This section demonstrates feature engineering patterns in BigQuery SQL.

We'll simulate this with pandas first (runs anywhere), then show the equivalent BigQuery SQL.

In [ ]:
# --- Create a realistic dataset for feature engineering ---

np.random.seed(42)
n = 1000

user_ids = [f'user_{i:04d}' for i in np.random.randint(1, 101, n)]
timestamps = pd.date_range('2024-01-01', periods=n, freq='2h')
amounts = np.random.lognormal(3, 1, n).round(2)
# Introduce some missing values
amounts[np.random.choice(n, 50, replace=False)] = np.nan
regions = np.random.choice(['US-East', 'US-West', 'EU', 'APAC'], n)
device_types = np.random.choice(['mobile', 'desktop', 'tablet'], n, p=[0.5, 0.35, 0.15])
# Binary label (imbalanced: 90% negative, 10% positive)
labels = np.random.choice([0, 1], n, p=[0.9, 0.1])

df = pd.DataFrame({
    'user_id': user_ids,
    'event_timestamp': timestamps,
    'amount': amounts,
    'region': regions,
    'device_type': device_types,
    'label': labels
})

print(f"Dataset: {n} rows, {df.columns.tolist()}")
print(f"Missing amounts: {df['amount'].isna().sum()}")
print(f"Label distribution: {dict(df['label'].value_counts())}")
df.head()

In [ ]:
# --- Derived feature engineering (equivalent to BigQuery SQL) ---

df_features = df.copy()

# Time-based features
df_features['day_of_week'] = df_features['event_timestamp'].dt.dayofweek
df_features['hour_of_day'] = df_features['event_timestamp'].dt.hour
df_features['is_weekend'] = (df_features['day_of_week'] >= 5).astype(int)

# Per-user aggregates (window functions in BQ)
user_stats = df_features.groupby('user_id')['amount'].agg(
    user_total_spend='sum',
    user_avg_spend='mean',
    user_event_count='count'
).reset_index()

df_features = df_features.merge(user_stats, on='user_id', how='left')

# Derived ratios
df_features['spend_vs_avg'] = df_features['amount'] / df_features['user_avg_spend']

print("Derived features created:")
print("  - day_of_week, hour_of_day, is_weekend (time-based)")
print("  - user_total_spend, user_avg_spend, user_event_count (aggregates)")
print("  - spend_vs_avg (ratio)")
print()
df_features[['user_id', 'amount', 'day_of_week', 'hour_of_day',
             'user_avg_spend', 'spend_vs_avg', 'label']].head(10)

In [ ]:
# --- Equivalent BigQuery SQL (for reference) ---

bq_sql = """
-- BigQuery SQL: Derived features for ML
SELECT
  user_id,
  event_timestamp,
  amount,
  region,
  device_type,

  -- Time-based features
  EXTRACT(DAYOFWEEK FROM event_timestamp) AS day_of_week,
  EXTRACT(HOUR FROM event_timestamp) AS hour_of_day,
  IF(EXTRACT(DAYOFWEEK FROM event_timestamp) IN (1, 7), 1, 0) AS is_weekend,

  -- Per-user aggregate features (window functions)
  SUM(amount) OVER(PARTITION BY user_id) AS user_total_spend,
  AVG(amount) OVER(PARTITION BY user_id) AS user_avg_spend,
  COUNT(*) OVER(PARTITION BY user_id) AS user_event_count,

  -- Rolling 7-day average
  AVG(amount) OVER(
    PARTITION BY user_id
    ORDER BY event_timestamp
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ) AS avg_spend_7d,

  -- Derived ratio
  amount / NULLIF(AVG(amount) OVER(PARTITION BY user_id), 0) AS spend_vs_avg,

  label
FROM `project.dataset.raw_events`;
"""

print(bq_sql)

---
## 3. Handle Missing Values & Class Imbalance

In [ ]:
# --- Missing value handling strategies ---

print("=== Missing Value Analysis ===")
print(f"Total rows: {len(df_features)}")
print(f"Missing amounts: {df_features['amount'].isna().sum()} ({df_features['amount'].isna().mean()*100:.1f}%)")
print()

# Strategy 1: Drop rows with missing values
df_dropped = df_features.dropna(subset=['amount'])
print(f"Strategy 1 - Drop: {len(df_dropped)} rows remaining")

# Strategy 2: Impute with median
median_amount = df_features['amount'].median()
df_imputed = df_features.copy()
df_imputed['amount_imputed'] = df_imputed['amount'].fillna(median_amount)
df_imputed['amount_is_missing'] = df_imputed['amount'].isna().astype(int)
print(f"Strategy 2 - Median impute: median = {median_amount:.2f}, added missing indicator")

# Strategy 3: Cap outliers (IQR method)
q1 = df_features['amount'].quantile(0.25)
q3 = df_features['amount'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_count = ((df_features['amount'] < lower_bound) | (df_features['amount'] > upper_bound)).sum()
df_imputed['amount_capped'] = df_imputed['amount_imputed'].clip(lower_bound, upper_bound)

print(f"Strategy 3 - IQR capping: bounds [{lower_bound:.2f}, {upper_bound:.2f}], {outlier_count} outliers capped")
print()

# Show the BigQuery SQL equivalent
print("-- BigQuery SQL equivalent:")
print("-- IFNULL(amount, (SELECT APPROX_QUANTILES(amount, 2)[OFFSET(1)] FROM table)) AS amount_imputed")
print("-- IF(amount IS NULL, 1, 0) AS amount_is_missing")
print("-- GREATEST(LEAST(amount, q3+1.5*iqr), q1-1.5*iqr) AS amount_capped")

In [ ]:
# --- Class imbalance handling ---

print("=== Class Imbalance Analysis ===")
print(f"Label distribution:")
print(df_features['label'].value_counts())
print(f"Imbalance ratio: {df_features['label'].value_counts()[0] / df_features['label'].value_counts()[1]:.1f}:1")
print()

# Strategy 1: Oversampling minority class
minority = df_features[df_features['label'] == 1]
majority = df_features[df_features['label'] == 0]

# Oversample minority to match majority count
minority_oversampled = minority.sample(n=len(majority), replace=True, random_state=42)
df_oversampled = pd.concat([majority, minority_oversampled]).sample(frac=1, random_state=42)
print(f"After oversampling: {dict(df_oversampled['label'].value_counts())}")

# Strategy 2: Undersampling majority class
majority_undersampled = majority.sample(n=len(minority), random_state=42)
df_undersampled = pd.concat([majority_undersampled, minority]).sample(frac=1, random_state=42)
print(f"After undersampling: {dict(df_undersampled['label'].value_counts())}")

# Strategy 3: Class weights (what BQML auto_class_weights does)
total = len(df_features)
n_classes = df_features['label'].nunique()
class_weights = {}
for cls in df_features['label'].unique():
    count = (df_features['label'] == cls).sum()
    class_weights[cls] = total / (n_classes * count)
print(f"Computed class weights (like auto_class_weights): {class_weights}")
print()
print("-- BQML equivalent:")
print("-- CREATE MODEL ... OPTIONS(auto_class_weights=TRUE)")

---
## 4. Partitioned Table Creation & Query Optimization

This section shows BigQuery partitioning/clustering SQL patterns. We'll show the SQL and simulate the cost savings with pandas.

In [ ]:
# --- Partitioning and Clustering SQL (reference) ---

partition_sql = """
-- Create a partitioned and clustered table
CREATE OR REPLACE TABLE `project.dataset.ml_features`
PARTITION BY DATE(event_timestamp)
CLUSTER BY user_id, region
AS
SELECT
  user_id,
  event_timestamp,
  region,
  device_type,
  IFNULL(amount, 0.0) AS amount,
  IF(amount IS NULL, 1, 0) AS amount_is_missing,
  EXTRACT(DAYOFWEEK FROM event_timestamp) AS day_of_week,
  EXTRACT(HOUR FROM event_timestamp) AS hour_of_day,
  label
FROM `project.dataset.raw_events`;


-- Set partition expiration (auto-delete data older than 365 days)
ALTER TABLE `project.dataset.ml_features`
SET OPTIONS (partition_expiration_days=365);
"""

print(partition_sql)

In [ ]:
# --- Simulate partition pruning cost savings ---

# Create a larger dataset spanning 1 year
np.random.seed(42)
n_large = 100_000
dates = pd.date_range('2023-01-01', '2024-12-31', periods=n_large)
df_large = pd.DataFrame({
    'event_date': dates.date,
    'amount': np.random.lognormal(3, 1, n_large),
    'user_id': [f'user_{i:04d}' for i in np.random.randint(1, 1001, n_large)]
})
df_large['event_date'] = pd.to_datetime(df_large['event_date'])

# Simulate bytes per row (~100 bytes)
bytes_per_row = 100
total_bytes = n_large * bytes_per_row
total_gb = total_bytes / 1e9

# Without partitioning: scan all data
cost_full = (total_gb / 1000) * 6.25

# With partitioning: scan only last 90 days
cutoff = df_large['event_date'].max() - timedelta(days=90)
rows_pruned = (df_large['event_date'] >= cutoff).sum()
pruned_gb = (rows_pruned * bytes_per_row) / 1e9
cost_pruned = (pruned_gb / 1000) * 6.25

print("=== Partition Pruning Cost Simulation ===")
print(f"Total dataset: {n_large:,} rows, ~{total_gb:.2f} GB")
print(f"Full scan cost: ${cost_full:.4f} ({total_gb:.2f} GB scanned)")
print(f"Pruned scan (90 days): ${cost_pruned:.4f} ({pruned_gb:.2f} GB scanned)")
print(f"Savings: {(1 - cost_pruned/cost_full)*100:.1f}%")
print()

# Query optimization SQL
print("-- Optimized query with partition pruning:")
print("SELECT * FROM `project.dataset.ml_features`")
print("WHERE event_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 90 DAY);")
print()
print("-- BAD: This prevents partition pruning!")
print("SELECT * FROM `project.dataset.ml_features`")
print("WHERE EXTRACT(YEAR FROM event_timestamp) = 2024;")

---
## 5. BQML: Engineered Features vs Raw Features

We'll compare model performance with raw features vs engineered features using sklearn locally. The BigQuery SQL equivalents are shown alongside.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Prepare dataset (drop rows with NaN for clean comparison)
df_ml = df_imputed.dropna(subset=['amount_imputed']).copy()

# --- Model 1: Raw features only ---
le_region = LabelEncoder()
le_device = LabelEncoder()

raw_features = pd.DataFrame({
    'amount': df_ml['amount_imputed'],
    'region': le_region.fit_transform(df_ml['region']),
    'device_type': le_device.fit_transform(df_ml['device_type'])
})

X_raw = raw_features.values
y = df_ml['label'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

scaler_raw = StandardScaler()
X_train_raw_s = scaler_raw.fit_transform(X_train_raw)
X_test_raw_s = scaler_raw.transform(X_test_raw)

model_raw = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model_raw.fit(X_train_raw_s, y_train)
y_pred_raw = model_raw.predict(X_test_raw_s)
y_proba_raw = model_raw.predict_proba(X_test_raw_s)[:, 1]

print("=== Model 1: Raw Features Only ===")
print(f"Features: amount, region (encoded), device_type (encoded)")
print(f"Accuracy: {accuracy_score(y_test, y_pred_raw):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_proba_raw):.4f}")
print()

In [ ]:
# --- Model 2: Engineered features ---

eng_features = pd.DataFrame({
    'amount': df_ml['amount_imputed'],
    'amount_capped': df_ml['amount_capped'],
    'amount_is_missing': df_ml['amount_is_missing'],
    'region': le_region.transform(df_ml['region']),
    'device_type': le_device.transform(df_ml['device_type']),
    'day_of_week': df_ml['day_of_week'],
    'hour_of_day': df_ml['hour_of_day'],
    'is_weekend': df_ml['is_weekend'],
    'user_total_spend': df_ml['user_total_spend'],
    'user_avg_spend': df_ml['user_avg_spend'],
    'user_event_count': df_ml['user_event_count'],
    'spend_vs_avg': df_ml['spend_vs_avg'].fillna(0)
})

X_eng = eng_features.values

X_train_eng, X_test_eng, _, _ = train_test_split(
    X_eng, y, test_size=0.2, random_state=42, stratify=y
)

scaler_eng = StandardScaler()
X_train_eng_s = scaler_eng.fit_transform(X_train_eng)
X_test_eng_s = scaler_eng.transform(X_test_eng)

model_eng = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model_eng.fit(X_train_eng_s, y_train)
y_pred_eng = model_eng.predict(X_test_eng_s)
y_proba_eng = model_eng.predict_proba(X_test_eng_s)[:, 1]

print("=== Model 2: Engineered Features ===")
print(f"Features: {list(eng_features.columns)}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_eng):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_proba_eng):.4f}")
print()

In [ ]:
# --- Comparison Summary ---

print("=" * 55)
print("  Feature Engineering Impact Comparison")
print("=" * 55)
print(f"{'Metric':<20} {'Raw Features':>15} {'Engineered':>15}")
print("-" * 55)
print(f"{'Num Features':<20} {X_raw.shape[1]:>15} {X_eng.shape[1]:>15}")
print(f"{'Accuracy':<20} {accuracy_score(y_test, y_pred_raw):>15.4f} {accuracy_score(y_test, y_pred_eng):>15.4f}")
print(f"{'ROC AUC':<20} {roc_auc_score(y_test, y_proba_raw):>15.4f} {roc_auc_score(y_test, y_proba_eng):>15.4f}")
print("=" * 55)
print()

auc_improvement = roc_auc_score(y_test, y_proba_eng) - roc_auc_score(y_test, y_proba_raw)
print(f"AUC Improvement from feature engineering: {auc_improvement:+.4f}")
print()
print("Key takeaway: Feature engineering (time features, aggregates,")
print("missing indicators, outlier capping) typically improves model")
print("performance even with the same algorithm.")

In [ ]:
# --- BQML equivalent SQL (for reference) ---  # REQUIRES GCP

bqml_comparison_sql = """
-- Model 1: Raw features
CREATE OR REPLACE MODEL `project.dataset.model_raw`
OPTIONS(
  model_type='LOGISTIC_REG',
  auto_class_weights=TRUE,
  input_label_cols=['label']
) AS
SELECT
  amount,
  region,
  device_type,
  label
FROM `project.dataset.raw_events`
WHERE event_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 90 DAY);


-- Model 2: Engineered features with TRANSFORM clause
CREATE OR REPLACE MODEL `project.dataset.model_engineered`
TRANSFORM(
  IFNULL(amount, 0.0) AS amount_imputed,
  IF(amount IS NULL, 1, 0) AS amount_is_missing,
  ML.BUCKETIZE(amount, [10, 25, 50, 100, 200]) AS amount_bucket,
  ML.FEATURE_CROSS(STRUCT(region, device_type)) AS region_device,
  EXTRACT(DAYOFWEEK FROM event_timestamp) AS day_of_week,
  EXTRACT(HOUR FROM event_timestamp) AS hour_of_day,
  AVG(amount) OVER(PARTITION BY user_id) AS user_avg_spend,
  COUNT(*) OVER(PARTITION BY user_id) AS user_event_count,
  label
)
OPTIONS(
  model_type='LOGISTIC_REG',
  auto_class_weights=TRUE,
  input_label_cols=['label']
) AS
SELECT * FROM `project.dataset.raw_events`
WHERE event_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 90 DAY);


-- Compare model performance
SELECT
  'raw' AS model,
  roc_auc, accuracy, log_loss
FROM ML.EVALUATE(MODEL `project.dataset.model_raw`)
UNION ALL
SELECT
  'engineered' AS model,
  roc_auc, accuracy, log_loss
FROM ML.EVALUATE(MODEL `project.dataset.model_engineered`);
"""

print(bqml_comparison_sql)

---
## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **Apache Beam** | PCollection + ParDo/DoFn + GroupByKey. DirectRunner for local, DataflowRunner for GCP |
| **Derived Features** | Time features, aggregates, ratios all doable in BigQuery SQL or Beam |
| **Missing Values** | Impute with median + add missing indicator. In BQML: `IFNULL()` + indicator column |
| **Class Imbalance** | BQML: `auto_class_weights=TRUE`. Or oversample/undersample in SQL |
| **Partitioning** | Partition by date, cluster by query columns. Always filter on partition column |
| **Feature Engineering** | Use BQML `TRANSFORM` clause. Raw vs engineered features = measurable AUC improvement |